# Can a pre-treatment biopsy image predict response to chemotherapy?
**T-CAIREM Datathon - Team 2 (Pink)**

Outcome: **complete** vs **partial** response to neoadjuvant chemotherapy (NAC).
Run the cells in order. Part 3 needs the GPU; its results are saved, so it only runs once.

## 0. Settings

In [ ]:
DATA_PATH = "Master_Data_Sheet.csv"   # clinical spreadsheet
WSI_DIR   = "T-CAIREM"                # one folder per patient: T-CAIREM/NAC-1/xxx.tif
OUT_DIR   = "/home/jupyter"           # the gcs data folder is read-only, so save here
SEED      = 42
TARGET    = "complete"                # 1 = complete response, 0 = partial

## 1. Setup

In [ ]:
import os, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import mannwhitneyu, chi2_contingency
from sklearn.model_selection import (train_test_split, cross_val_score, cross_val_predict,
                                     StratifiedKFold, RepeatedStratifiedKFold)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn import metrics
np.random.seed(SEED)

## 2. Load and clean the clinical data
- Column names have extra spaces, so we tidy them.
- The file codes **0 = complete, 1 = partial**. We flip it so **1 = complete** (the outcome we predict). This does not change any result.
- LVI has spelling variants, so we unify them.

In [ ]:
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.split().str.join(" ")
df[TARGET] = (df["NAC Response (0=complete; 1=partial)"] == 0).astype(int)

df["LVI"] = (df["LVI"].astype(str).str.strip().str.lower()
             .replace({"inderterminate": "unknown", "indeterminate": "unknown", "not reported": "unknown"}))

resp = df.set_index("NAC")[TARGET].map({1: "complete", 0: "partial"})   # response per patient
print(df.shape)
print(df[TARGET].value_counts().rename({1: "complete", 0: "partial"}))

## 3. Explore the data (EDA)
### 3.1 Response by subtype

In [ ]:
rate = df.groupby("Type")[TARGET].agg(["mean", "size"]).round(2)
print(rate)
rate["mean"].plot.bar(ylabel="Complete response rate"); plt.show()

### 3.2 Numeric variables vs response (Mann-Whitney test)

In [ ]:
NUM_EDA = ["Age", "ER (%)", "PR (%)", "cT (Size, mm)"]
fig, ax = plt.subplots(1, 4, figsize=(14, 3))
for a, c in zip(ax, NUM_EDA):
    sns.boxplot(x=df[TARGET], y=df[c], ax=a)
    p = mannwhitneyu(df.loc[df[TARGET] == 1, c], df.loc[df[TARGET] == 0, c]).pvalue
    a.set(title=f"{c}  (p = {p:.3f})", xlabel="Response (0 = partial, 1 = complete)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/eda_boxplots.png", dpi=200); plt.show()

### 3.3 Categorical variables vs response (chi-square test)

In [ ]:
for c in ["Type", "HER2 (0=neg; 1=pos)", "Histology (0=IDC; 1=ILC; 2=IMC)", "LVI",
          "Grade Nottingham", "cT (TNM, ajcc V8)", "cN (TNM, ajcc V8)"]:
    tab = pd.crosstab(df[c], df[TARGET])
    print(f"\n{c}   p = {chi2_contingency(tab)[1]:.3f}")
    print(pd.crosstab(df[c], df[TARGET], normalize="index").round(2).assign(n=tab.sum(axis=1)))

### 3.4 Correlation between numeric variables (finds near-duplicates)

In [ ]:
corr_cols = ["Age", "ER (%)", "PR (%)", "cT (Size, mm)", "Grade Nottingham",
             "cT (TNM, ajcc V8)", "cN (TNM, ajcc V8)"]
sns.heatmap(df[corr_cols].corr(method="spearman"), annot=True, fmt=".2f",
            cmap="coolwarm", vmin=-1, vmax=1)
plt.savefig(f"{OUT_DIR}/eda_correlation.png", dpi=200, bbox_inches="tight"); plt.show()

## 4. Choose variables (pre-treatment only)
- **Kept (9):** clinical = age, tumour size, nodal stage; pathological = ER %, HER2, grade, histology, LVI; plus subtype.
- **Removed:** post-chemo variables (RCB, surgery, adjuvant therapy, survival) because they leak the answer;
  PR % (r = 0.90 with ER %) and cT stage (r = 0.84 with size) as near-duplicates; IDs and scanner details.
- **Rare groups merged:** grade 1 -> "1-2" (only 4 cases), cN3 -> "2+" (only 1 case).

In [ ]:
df["Grade Nottingham"] = df["Grade Nottingham"].clip(lower=2)      # 1-2 vs 3
df["cN (TNM, ajcc V8)"] = df["cN (TNM, ajcc V8)"].clip(upper=2)    # 0, 1, 2+

CONT = ["Age", "ER (%)", "cT (Size, mm)"]
ORD  = ["Grade Nottingham", "cN (TNM, ajcc V8)"]
CAT  = ["Type", "HER2 (0=neg; 1=pos)", "Histology (0=IDC; 1=ILC; 2=IMC)", "LVI"]
df[CAT] = df[CAT].astype(str)

X, y = df[CONT + ORD + CAT], df[TARGET]

def make_pre(cat=CAT):
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), CONT + ORD),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])

pre = make_pre()
lr = lambda: LogisticRegression(max_iter=5000)

## 5. Clinical + pathological model
80% training / 20% test. Three methods compared with 5-fold CV on the training set.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

models = {"Logistic": lr(),
          "Random forest": RandomForestClassifier(500, random_state=SEED),
          "Gradient boosting": HistGradientBoostingClassifier(random_state=SEED)}
for name, m in models.items():
    auc = cross_val_score(Pipeline([("pre", pre), ("m", m)]), X_tr, y_tr, cv=5, scoring="roc_auc")
    print(f"{name}: CV AUC = {auc.mean():.2f} +/- {auc.std():.2f}")

In [ ]:
best = Pipeline([("pre", pre), ("m", lr())]).fit(X_tr, y_tr)
p = best.predict_proba(X_te)[:, 1]
print("Test AUC:", round(metrics.roc_auc_score(y_te, p), 2))
metrics.RocCurveDisplay.from_predictions(y_te, p); plt.show()

imp = permutation_importance(best, X_te, y_te, scoring="roc_auc", n_repeats=20, random_state=SEED)
pd.Series(imp.importances_mean, index=X.columns).sort_values().plot.barh()
plt.xlabel("Drop in AUC when shuffled")
plt.savefig(f"{OUT_DIR}/clinical_importance.png", dpi=200, bbox_inches="tight"); plt.show()

## 6. Checks on the clinical model
Repeated 5-fold CV on all patients (same splits are reused later for the image comparison).
- **Without Luminal A:** no Luminal A patient responded, which makes the task look easier than it is.
- **Without LVI:** LVI may have been assessed on the surgery specimen (after chemo); this shows how much it matters.

In [ ]:
cv  = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=SEED)
sub = (df["Type"] != "Luminal A").values

def auc_cv(model, Xd, yd):
    return cross_val_score(model, Xd, yd, cv=cv, scoring="roc_auc")

no_lvi = [c for c in CAT if c != "LVI"]
for name, model in {"Clinical": Pipeline([("pre", make_pre()), ("m", lr())]),
                    "Clinical without LVI": Pipeline([("pre", make_pre(no_lvi)), ("m", lr())])}.items():
    print(f"{name:22s} All: {auc_cv(model, X, y).mean():.2f}   "
          f"Without Luminal A: {auc_cv(model, X[sub], y[sub]).mean():.2f}")

## 7. Image features (GPU, run once)
For each slide: find tissue on a small preview -> cut up to 300 patches of 224 px at 20x ->
**Phikon** (published pathology model, used as-is) turns each patch into 768 numbers -> average per patient.
No outcome information is used here.

In [ ]:
# Run once, then restart the kernel:
# !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu124
# !pip install -q openslide-python openslide-bin transformers

In [ ]:
import torch, openslide
from transformers import ViTModel
from torchvision import transforms
print("GPU available:", torch.cuda.is_available())

enc = ViTModel.from_pretrained("owkin/phikon", add_pooling_layer=False).cuda().half().eval()
tf = transforms.Compose([transforms.ToTensor(),
                         transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])

def get_patches(path, n=300, size=224):
    """Random tissue patches at ~20x. Returns patches, their positions on a small preview, the preview, the slide."""
    s = openslide.OpenSlide(path)
    W, H = s.dimensions
    mpp = float(s.properties.get("openslide.mpp-x", 0.2))        # microns per pixel (0.2 if not stored)
    step = int(size * 0.5 / mpp)                                  # patch size in full-resolution pixels
    small = np.array(s.get_thumbnail((W // step, H // step)).convert("RGB"))   # 1 pixel = 1 patch
    sx, sy = W / small.shape[1], H / small.shape[0]
    ys, xs = np.where((small.mean(2) < 220) & (small.max(2) - small.min(2) > 20))  # tissue, not background
    pick = np.random.default_rng(0).permutation(len(xs))[:n]
    xs, ys = xs[pick], ys[pick]
    lvl = s.get_best_level_for_downsample(step / size * 1.01)
    rs = int(step / s.level_downsamples[lvl])
    patches = [s.read_region((int(x * sx), int(yy * sy)), lvl, (rs, rs)).convert("RGB").resize((size, size))
               for x, yy in zip(xs, ys)]
    return patches, xs, ys, small, s

def encode(patches):
    """Phikon features (768 per patch)."""
    with torch.no_grad():
        return torch.cat([enc(pixel_values=torch.stack([tf(p) for p in patches[b:b + 64]]).cuda().half())
                          .last_hidden_state[:, 0].float().cpu()
                          for b in range(0, len(patches), 64)]).numpy()

print(encode(get_patches(glob.glob(f"{WSI_DIR}/NAC-1/*.tif")[0])[0]).shape)   # test one slide: (300, 768)

In [ ]:
# All slides. Saved after each one, so rerunning continues where it stopped.
out = f"{OUT_DIR}/image_features.csv"
done = pd.read_csv(out, index_col=0) if os.path.exists(out) else pd.DataFrame()
for path in sorted(glob.glob(f"{WSI_DIR}/NAC-*/*.tif")):
    case = path.split("/")[-2]                      # folder name = NAC id
    if case in done.index:
        continue
    row = pd.DataFrame([encode(get_patches(path)[0]).mean(0)], index=[case]).add_prefix("img_")
    done = pd.concat([done, row]); done.to_csv(out)
    print(case, "done")
print("Patients with image features:", len(done))

## 8. Does the image add information?
Clinical vs image vs both. Logistic regression, repeated 5-fold CV (same folds for all models).
Image features are compressed to 10 components **inside each fold** to avoid overfitting.

In [ ]:
img = pd.read_csv(f"{OUT_DIR}/image_features.csv", index_col=0)
I = list(img.columns)
XI = pd.concat([X, img.loc[df["NAC"]].set_axis(X.index)], axis=1)

img_pre = Pipeline([("sc", StandardScaler()), ("pca", PCA(10))])
compare = {"Clinical": make_pre(),
           "Image": ColumnTransformer([("img", img_pre, I)]),
           "Clinical + Image": ColumnTransformer(make_pre().transformers + [("img", img_pre, I)])}

res = {}
for name, p in compare.items():
    model = Pipeline([("pre", p), ("m", lr())])
    res[name] = (auc_cv(model, XI, y), auc_cv(model, XI[sub], y[sub]))
    print(f"{name:17s} All: {res[name][0].mean():.2f}   Without Luminal A: {res[name][1].mean():.2f}")

gain = res["Clinical + Image"][1] - res["Clinical"][1]
print(f"\nGain without Luminal A = {gain.mean():.3f} (SD {gain.std():.3f}); "
      f"images help in {(gain > 0).mean():.0%} of folds")

## 9. What do the images capture?

In [ ]:
# 9.1 Example slide per subtype, with that patient's response from the spreadsheet
examples = {"Luminal A": "NAC-1", "Luminal B": "NAC-41", "TNBC": "NAC-80", "HER2+": "NAC-120"}
fig, ax = plt.subplots(1, 4, figsize=(16, 5))
for a, (t, case) in zip(ax, examples.items()):
    a.imshow(openslide.OpenSlide(glob.glob(f"{WSI_DIR}/{case}/*.tif")[0]).get_thumbnail((800, 800)))
    a.set_title(f"{t}\nresponse: {resp[case]}"); a.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_slides.png", dpi=200); plt.show()

In [ ]:
# 9.2 Each patient as a dot, placed by image similarity
emb = PCA(2).fit_transform(StandardScaler().fit_transform(img))
lab = df.set_index("NAC").loc[img.index]
lab["Response"] = lab[TARGET].map({1: "complete", 0: "partial"})
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
for a, col in zip(ax, ["Type", "Response", "Digital Pathology System"]):
    for v in lab[col].unique():
        m = (lab[col] == v).values
        a.scatter(emb[m, 0], emb[m, 1], label=v, s=20)
    a.set_title(f"Image features coloured by {col}"); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_map.png", dpi=200); plt.show()

In [ ]:
# 9.3 Can the image alone tell the subtype?
lab_t = lab["Type"]
pred = cross_val_predict(make_pipeline(StandardScaler(), PCA(20), lr()), img, lab_t,
                         cv=StratifiedKFold(5, shuffle=True, random_state=SEED))
print("Balanced accuracy:", round(metrics.balanced_accuracy_score(lab_t, pred), 2), "(chance = 0.25)")
metrics.ConfusionMatrixDisplay.from_predictions(lab_t, pred, normalize="true", cmap="Blues")
plt.savefig(f"{OUT_DIR}/fig_subtype.png", dpi=200, bbox_inches="tight"); plt.show()

## 10. Heatmaps: where on the slide does the model look?
Each patch is scored by a model trained **without** this patient (illustration only:
the model was trained on patient averages and applied to single patches).

In [ ]:
def heatmap(case):
    path = glob.glob(f"{WSI_DIR}/{case}/*.tif")[0]
    patches, xs, ys, small, s = get_patches(path)
    yi = df.set_index("NAC").loc[img.index, TARGET]
    train = img.index != case                                    # leave this patient out
    clf = make_pipeline(StandardScaler(), PCA(10), lr()).fit(img[train], yi[train])
    score = clf.predict_proba(pd.DataFrame(encode(patches), columns=img.columns))[:, 1]

    big = s.get_thumbnail((2000, 2000)); k = big.size[0] / small.shape[1]
    fig, ax = plt.subplots(1, 2, figsize=(14, 6))
    ax[0].imshow(big); ax[0].set_title(f"{case} biopsy (actual response: {resp[case]})"); ax[0].axis("off")
    ax[1].imshow(big)
    sc = ax[1].scatter(xs * k, ys * k, c=score, cmap="coolwarm", s=15, vmin=0, vmax=1)
    plt.colorbar(sc, ax=ax[1], label="Predicted chance of complete response"); ax[1].axis("off")
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_heatmap_{case}.png", dpi=200); plt.show()

    order = np.argsort(score)
    fig, ax = plt.subplots(2, 5, figsize=(12, 5))
    for a, i in zip(ax[0], order[-5:][::-1]): a.imshow(patches[i]); a.axis("off")
    for a, i in zip(ax[1], order[:5]): a.imshow(patches[i]); a.axis("off")
    ax[0, 0].set_title("Most responder-like", loc="left"); ax[1, 0].set_title("Least responder-like", loc="left")
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig_patches_{case}.png", dpi=200); plt.show()

heatmap("NAC-120")   # HER2+, complete response
heatmap("NAC-1")     # Luminal A, partial response